# Small Multiples

**DS4DH · Module 10 — Data Visualization and Communication**

*Technique:* Repeated panels, the shared-axis rule, and reference lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/10c_small_multiples.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A small multiple is the same chart repeated across categories. It works because
the reader learns the chart once and then only has to compare.

It has exactly one non-negotiable rule: **every panel shares the same axes.**
Break it and the comparison the layout invites becomes false, which is worse than
not drawing the comparison at all.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Total'])

# The failure first. Note sharex/sharey are NOT set.
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, city in zip(axes.ravel(), CITIES):
    s = d[d['cma'] == city]['Total']
    ax.hist(s, bins=12, color='#3DA5D9', edgecolor='white', linewidth=0.5)
    ax.set_title(f'{city} (n={len(s)})')
fig.suptitle('WRONG — independent axes make these panels non-comparable')
plt.tight_layout()
plt.show()

print('Axis ranges actually used:')
for city in CITIES:
    s = d[d['cma'] == city]['Total']
    print(f'  {city:<12}{s.min():>6.1f} to {s.max():>6.1f}')

In [ ]:
# The fix: shared axes, plus a common reference line.
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, city in zip(axes.ravel(), CITIES):
    s = d[d['cma'] == city]['Total']
    ax.hist(s, bins=np.linspace(d['Total'].min(), d['Total'].max(), 16),
            color='#3DA5D9', edgecolor='white', linewidth=0.5)
    ax.axvline(30, color='#333', ls=':', lw=1.2)
    ax.axvline(s.median(), color='#E8663D', lw=2)
    ax.set_title(f'{city}  (n={len(s)}, median {s.median():.1f})')
    ax.set_xlabel('Total STIR (%)')
fig.suptitle('Housing burden by CMA — shared axes, 30% line, city median in orange')
plt.tight_layout()
plt.show()

With shared axes the comparison is immediate and true: you can see which cities
sit further right, which are more spread out, and how each relates to the 30%
threshold — none of which the first figure supported.

In [ ]:
# Densities work better than counts when the group sizes differ this much.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharex=True, sharey=True)
grid = np.linspace(d['Total'].min() - 2, d['Total'].max() + 2, 300)
overall = stats.gaussian_kde(d['Total'].values)

for ax, city in zip(axes, CITIES):
    s = d[d['cma'] == city]['Total'].values
    ax.plot(grid, overall(grid), color='#999', lw=1.2, ls='--', label='all CSDs')
    if len(s) >= 5:
        ax.plot(grid, stats.gaussian_kde(s)(grid), color='#E8663D', lw=2.2, label=city)
    ax.axvline(30, color='#333', ls=':', lw=1)
    ax.set_title(f'{city} (n={len(s)})')
    ax.set_xlabel('Total STIR (%)')
axes[0].set_ylabel('density')
axes[0].legend(fontsize=8)
fig.suptitle('Each city against the same national-sample backdrop')
plt.tight_layout()
plt.show()

Repeating the grey overall curve in every panel is a small multiples technique
worth knowing: it gives each panel its own local baseline, so the reader compares
against a constant rather than holding three other panels in memory.

In [ ]:
# Small multiples of a relationship, not just a distribution.
dd = d.dropna(subset=['tot_income'])
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)
for ax, city in zip(axes, CITIES):
    g = dd[dd['cma'] == city]
    ax.scatter(dd['tot_income'], dd['Total'], s=14, color='#DDD')     # context
    ax.scatter(g['tot_income'], g['Total'], s=20, color='#E8663D')    # focus
    ax.axhline(30, color='#333', ls=':', lw=1)
    ax.set_title(f'{city} (n={len(g)})')
    ax.set_xlabel('income ($)')
axes[0].set_ylabel('Total STIR (%)')
fig.suptitle('Income vs burden — each city highlighted against all CSDs in grey')
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Remove `sharex=True, sharey=True` from the last figure and re-run.

The grey context points now sit differently in each panel. Explain precisely what
a reader would wrongly conclude from that version.

### 🔧 Your turn 2

Rebuild the four-panel histogram sorted by median burden rather than in the fixed
`CITIES` order.

Ordering panels by the quantity being shown is a small multiples convention. What
does it add, and when would you *not* want it?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With independent axes, the grey context cloud is rescaled per
panel, so a reader comparing panels sees the *same* set of CSDs occupying
different positions — and would conclude that the income–burden relationship
differs between cities when the difference is entirely an artefact of the axis
limits. This is the worst version of the failure, because the shared grey backdrop
actively signals comparability that has been removed.

**Your turn 2.** Sorting by the displayed quantity turns the panel order itself
into information — the reader gets the ranking for free, before reading any
values. You would not want it when the categories have a meaningful natural order
(time, geography from west to east, age bands), because overriding that order
costs the reader more than the ranking gains, or when panel order needs to stay
stable across a series of figures so readers can compare between them.

</details>

## Where this stops

Module 10's rule set, in three lines: share the axes, put the reference line in
every panel, and say in the caption what you filtered.

Next: Module 11 asks what any of this entitles you to recommend.